In [ ]:
from project_utils import load_table, input_path, output_path, calculateCI, mean_error, standard_error, percentile_column, positive_fraction, SEED
import numpy as np
np.random.seed(SEED)


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm


In [ ]:
df = load_table('papers')


In [ ]:
df['Field'] = df['Field'].apply(lambda x:int(x//10))

def ref_classify(x):
    if x <= 10:
        return  '1-10'
    elif x <= 20:
        return  '11-20'
    elif x <= 30:
        return  '21-30'
    else:
        return  '>30'
df['binref'] = df['outdegree'].apply(ref_classify)

def ref_classify2(x):
    if pd.isna(x):
        return np.nan
    if x >= 1980 and x<1990:
        return  '1980-1989'
    elif x >= 1990 and x<2000:
        return  '1990-1999'
    elif x >= 2000 and x<2010:
        return  '2000-2009'
    elif x >= 2010:
        return '2010+'
    else:
        return  '<1980'
df['binyear'] = df['year'].apply(ref_classify2)


In [ ]:
df.columns


# Statistic Tables


In [ ]:
columns = ['outdegree','indegree','ni','nj','nk','rnk','CD','D','RD']

dff = df[columns].describe().T
dff['skewness'] = df[columns].skew()
dff['kurt'] = df[columns].kurt()
dff.to_excel(output_path('figure_3_parameters_6.xlsx'))


In [ ]:
topv = np.percentile(df['indegree'],99)
df1 = df[df['indegree']>topv]
df2 = df[df['indegree']<=topv]

dff = df1[columns].describe().T
dff['skewness'] = df1[columns].skew()
dff['kurt'] = df1[columns].kurt()
dff.to_excel(output_path('figure_3_parameters_top.xlsx'))

dff = df2[columns].describe().T
dff['skewness'] = df2[columns].skew()
dff['kurt'] = df2[columns].kurt()
dff.to_excel(output_path('figure_3_parameters_other.xlsx'))


In [ ]:
df.columns


# figures


In [ ]:
parameters = ['ni','nj','nk','rnk']
parameter_labels = ['DC', 'CC', 'RC', r'$\widehat{RC}$']


In [ ]:
ds = [df[p].mean() for p in parameters]
stds = [df[p].std() for p in parameters]


In [ ]:
topv = np.percentile(df['indegree'],99)
ds1 = [df[df['indegree']>topv][p].mean() for p in parameters]
stds1 = [df[df['indegree']>topv][p].std() for p in parameters]
ds2 = [df[df['indegree']<=topv][p].mean() for p in parameters]
stds2 = [df[df['indegree']<=topv][p].std() for p in parameters]


In [ ]:
c = ['#1A5276','#BA4A00','#138D75','#138D75']*3
x  = [1,2,3,4,6,7,8,9,11,12,13,14]


In [ ]:
f, ax = plt.subplots(figsize=(15, 7))
# x = [1,2,3,4]
plt.bar(x , ds+ds1+ds2 , yerr = stds+stds1+stds2 ,color = c,capsize = 15,hatch = ['','','','/']*3 , alpha = .9)

ax.set_yscale("log")
plt.ylim(1,10e3)
plt.yticks([10e-1,10e0,10e1,10e2,10e3,],fontsize=20)
plt.xticks(x,parameter_labels*3,fontsize=30,)
plt.ylabel('Mean Value',fontsize = 30)
plt.show()


In [ ]:
f, ax = plt.subplots(figsize=(15, 7))
d = ds+ds1+ds2
plt.bar(x , d ,color = c,capsize = 15,hatch = ['','','','/']*3 , alpha = .9)
for _ in range(len(d)):
    if d[_]>200:
        plt.text(x[_], 800-30 , str(d[_].round(1)),fontsize = 12,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x[_], d[_]+10 , str(d[_].round(1)),fontsize = 12,c = 'black', ha='center', fontweight='bold')
plt.ylim(0,800)
plt.yticks(fontsize=20)
plt.xticks(x,parameter_labels*3,fontsize=30,)
plt.ylabel('Mean Value',fontsize = 30)
plt.show()


In [ ]:
formula = 'ni ~ outdegree'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df).fit()
print(model.summary())


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['outdegree'], df['ni'], c = 'black' , alpha = 0.01)
sns.regplot(data = df , y='ni' , x='outdegree', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False)
plt.ylim(0,100)
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Reference Count',fontsize = 30)
plt.ylabel('DC',fontsize = 30)
plt.show()


In [ ]:
formula = 'nj ~ outdegree'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df).fit()
print(model.summary())


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['outdegree'], df['nj'], c = 'black' , alpha = 0.01)
sns.regplot(data = df , y='nj' , x='outdegree', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False)
plt.ylim(0,100)
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Reference Count',fontsize = 30)
plt.ylabel('CC',fontsize = 30)
plt.show()


In [ ]:
formula = 'nk ~ outdegree'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df).fit()
print(model.summary())


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['outdegree'], df['nk'], c = 'black' , alpha = 0.01)
sns.regplot(data = df , y='nk' , x='outdegree', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False)
plt.ylim(0,100)
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Reference Count',fontsize = 30)
plt.ylabel('RC',fontsize = 30)
plt.show()


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['outdegree'], df['nk'], c = 'black' , alpha = 0.01)
sns.regplot(data = df , y='nk' , x='outdegree', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False)
plt.ylim(0,3000)
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Reference Count',fontsize = 30)
plt.ylabel('RC',fontsize = 30)
plt.show()


In [ ]:
formula = 'rnk ~ outdegree'
# fit the OLS model
model = sm.formula.ols(formula=formula, data=df).fit()
print(model.summary())


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(df['outdegree'], df['rnk'], c = 'black' , alpha = 0.01)
sns.regplot(data = df , y='rnk' , x='outdegree', x_ci = 90 , n_boot = 100 ,seed = 0 , color = '#922B21',scatter = False)
plt.ylim(0,100)
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Reference Count',fontsize = 30)
plt.ylabel(r'$\widehat{RC}$',fontsize = 30)
plt.show()


In [ ]:
yearlabel = ['1980-1989', '1990-1999', '2000-2009']
dff = df[df['binyear'].isin(yearlabel)]
f, ax = plt.subplots(figsize=(7, 3))
sns.barplot(data=dff, x="binyear", y='CD', order=yearlabel, errwidth=3, capsize=.2)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Publication Year',fontsize = 30)
plt.ylabel(r'DI',fontsize = 30)
plt.show()


In [ ]:
f, ax = plt.subplots(figsize=(7, 3))
sns.barplot(data=df, x="Field", y='CD' , errwidth=3, capsize=.2)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.xlabel('Subject Field',fontsize = 30)
plt.ylabel(r'DI',fontsize = 30)
plt.show()
